# Lab 02: LangFuse Setup & Deployment

**Goal:** Set up LangFuse SDK in mock mode (local JSON logging)
and configure environment variables.

**What you'll learn:**
- LangFuse architecture in mock mode
- How to implement a MockLangfuse class for local trace logging
- Environment variables needed for LangFuse client initialization
- The v4 observation pattern: start_observation / update / end / flush

In [ ]:
import os
import shutil
import textwrap

WORKDIR = "/tmp/k8s-lab-12-02"

if os.path.exists(WORKDIR):
    shutil.rmtree(WORKDIR)
os.makedirs(WORKDIR, exist_ok=True)

## Step 1: LangFuse Architecture (Mock Mode)

In [ ]:
print("LangFuse mock mode components:\n")
print("    LangChain App")
print("        \u2502")
print("        \u25bc")
print("    LangFuse SDK / CallbackHandler")
print("        \u2502")
print("        \u25bc")
print("    MockLangfuse (local JSON logging)")
print("        \u2502")
print("        \u25bc")
print("    JSON files (trace storage)")
print()
print("Mock mode: Same SDK patterns, data saved to local JSON files")
print("In production: Point LANGFUSE_HOST to a real LangFuse server")

## Step 2: MockLangfuse Reference

In [ ]:
mock_ref = textwrap.dedent("""\
    import json
    import os
    import uuid
    from datetime import datetime

    class MockLangfuse:
        \"\"\"Mock LangFuse v4 client that logs observations to local JSON files.\"\"\"

        def __init__(self, public_key, secret_key, host, output_dir=\"/tmp/langfuse-traces\"):
            self.public_key = public_key
            self.secret_key = secret_key
            self.host = host
            self.output_dir = output_dir
            os.makedirs(output_dir, exist_ok=True)
            self._traces = {}
            self._scores = []

        def auth_check(self):
            return bool(self.public_key and self.secret_key)

        @staticmethod
        def create_trace_id(seed=None):
            return uuid.uuid5(uuid.NAMESPACE_URL, seed).hex if seed else uuid.uuid4().hex

        def start_observation(self, name, as_type=\"span\", trace_id=None, **kwargs):
            trace_id = trace_id or self.create_trace_id()
            trace = self._traces.setdefault(trace_id, {
                \"trace_id\": trace_id,
                \"timestamp\": datetime.now().isoformat(),
                \"observations\": [],
            })
            obs = {\"name\": name, \"as_type\": as_type, **kwargs}
            trace[\"observations\"].append(obs)
            return MockObservation(obs, trace_id)

        def create_score(self, trace_id, name, value, comment=None):
            self._scores.append({\"trace_id\": trace_id, \"name\": name,
                                 \"value\": value, \"comment\": comment})

        def flush(self):
            output_file = os.path.join(self.output_dir, \"traces.json\")
            with open(output_file, \"w\") as f:
                json.dump({\"traces\": list(self._traces.values()),
                           \"scores\": self._scores}, f, indent=2)
            return len(self._traces)

        def get_traces(self):
            # Mock stand-in for the real client's langfuse.api.trace.list()
            output_file = os.path.join(self.output_dir, \"traces.json\")
            if os.path.exists(output_file):
                with open(output_file) as f:
                    return json.load(f)[\"traces\"]
            return []

    class MockObservation:
        def __init__(self, data, trace_id):
            self._data = data
            self.trace_id = trace_id

        def update(self, **kwargs):
            self._data.update(kwargs)
            return self

        def end(self):
            self._data[\"ended_at\"] = datetime.now().isoformat()
            return self
""")

print("Reference MockLangfuse class:\n")
for line in mock_ref.strip().split("\n"):
    print(f"    {line}")

with open(os.path.join(WORKDIR, "mock_langfuse_reference.py"), "w") as f:
    f.write(mock_ref)

## Step 3: Environment Variables

In [ ]:
print("Client-side (your Python app):\n")
env_vars = [
    ("LANGFUSE_PUBLIC_KEY",  "pk-lf-mock-...",            "API public key"),
    ("LANGFUSE_SECRET_KEY",  "sk-lf-mock-...",            "API secret key"),
    ("LANGFUSE_HOST",        "http://localhost:8000",      "LangFuse server URL (or mock)"),
]
for name, example, desc in env_vars:
    print(f"    {name:<25} {example:<30} # {desc}")

print("\nMock mode configuration:\n")
mock_vars = [
    ("output_dir",           "/tmp/langfuse-traces"),
    ("flush()",              "Writes traces to JSON file"),
    ("get_traces()",         "Reads back traces (stands in for api.trace.list())"),
]
for name, desc in mock_vars:
    print(f"    {name:<20} {desc}")

## TODO 1: MockLangfuse Configuration

Create a MockLangfuse setup with:
- MockLangfuse class with `__init__`, `auth_check`, `create_trace_id`,
  `start_observation`, `create_score`, `flush`, `get_traces`
- `start_observation(name, as_type=...)` appends to the trace's observations list
  and returns an observation with `update()` and `end()`
- `create_score()` records a score against a `trace_id`
- `flush()` writes all traces and scores to a JSON file
- `get_traces()` reads traces back from the JSON file

In [ ]:
# TODO: MockLangfuse setup for local trace logging (langfuse v4 shape)
# Include MockLangfuse with start_observation/create_score/flush/get_traces

todo1_code = textwrap.dedent("""\
    # TODO: MockLangfuse setup for local trace logging
    # Include MockLangfuse class with trace/flush/get_traces

""")

with open(os.path.join(WORKDIR, "mock_langfuse.py"), "w") as f:
    f.write(todo1_code)

In [ ]:
checks1 = [
    ("Has class MockLangfuse",       "class MockLangfuse" in todo1_code),
    ("Has __init__ method",          "__init__" in todo1_code),
    ("Has output_dir param",         "output_dir" in todo1_code),
    ("Has auth_check method",        "def auth_check" in todo1_code),
    ("Has create_trace_id method",   "def create_trace_id" in todo1_code),
    ("Has start_observation method", "def start_observation" in todo1_code),
    ("Has as_type parameter",        "as_type" in todo1_code),
    ("Has observations list",        "observations" in todo1_code),
    ("Has create_score method",      "def create_score" in todo1_code),
    ("Has flush method",             "def flush" in todo1_code),
    ("Has json.dump",                "json.dump" in todo1_code),
    ("Has get_traces method",        "def get_traces" in todo1_code),
    ("Has json.load",                "json.load" in todo1_code),
    ("Has MockObservation class",    "MockObservation" in todo1_code),
    ("Has update() and end()",       "def update" in todo1_code and "def end" in todo1_code),
]

score1 = sum(1 for _, ok in checks1 if ok)
print(f"Validating ({score1}/{len(checks1)}):\n")
for name, ok in checks1:
    print(f"    [{'PASS' if ok else 'FAIL'}] {name}")

## TODO 2: Client Environment Setup

Create the Python code to initialize LangFuse client.
Include: environment variables, `Langfuse()` client, health check.

In [ ]:
# TODO: LangFuse client initialization
# Set up env vars, create the client, verify with auth_check()

todo2_code = textwrap.dedent("""\
    # TODO: LangFuse client initialization
    # Set up env vars and create client

""")

with open(os.path.join(WORKDIR, "langfuse_client.py"), "w") as f:
    f.write(todo2_code)

In [ ]:
checks2 = [
    ("Has os import",              "import os" in todo2_code or "from os" in todo2_code),
    ("Has LANGFUSE_PUBLIC_KEY",    "LANGFUSE_PUBLIC_KEY" in todo2_code),
    ("Has LANGFUSE_SECRET_KEY",    "LANGFUSE_SECRET_KEY" in todo2_code),
    ("Has LANGFUSE_HOST",          "LANGFUSE_HOST" in todo2_code),
    ("Has Langfuse import",        "Langfuse" in todo2_code or "langfuse" in todo2_code),
    ("Has client creation",        "Langfuse(" in todo2_code),
    ("Has auth_check health probe", "auth_check" in todo2_code),
]

score2 = sum(1 for _, ok in checks2 if ok)
print(f"Validating ({score2}/{len(checks2)}):\n")
for name, ok in checks2:
    print(f"    [{'PASS' if ok else 'FAIL'}] {name}")

## Summary

In [ ]:
print("Key concepts:")
print("  1. Mock LangFuse: Same SDK patterns, traces saved to local JSON")
print("  2. MockLangfuse: start_observation(), create_score(), flush(), get_traces()")
print("  3. Client env vars: LANGFUSE_PUBLIC_KEY, SECRET_KEY, HOST")
print("  4. In production: Point LANGFUSE_HOST to a real LangFuse server")
print(f"\nTODO 1: {score1}/{len(checks1)} MockLangfuse checks passed")
print(f"TODO 2: {score2}/{len(checks2)} client setup checks passed")
print(f"\nFiles generated in {WORKDIR}/")

## Key Takeaways

- **Mock LangFuse** uses the same SDK patterns as production, but saves traces to local JSON files instead of a remote server
- **MockLangfuse class** mirrors the langfuse v4 surface -- `start_observation()`, `create_score()`, `flush()`, and `auth_check()` -- for local observability
- **Environment variables** (`LANGFUSE_PUBLIC_KEY`, `LANGFUSE_SECRET_KEY`, `LANGFUSE_HOST`) configure the LangFuse client
- **In production**, simply change `LANGFUSE_HOST` to point to a real LangFuse server -- no code changes needed
- The **observation pattern** (`start_observation(as_type="span"|"generation")` -> `update()` -> `end()`) captures the hierarchical structure of LLM calls within your application